**Utilizando ANOVA**

O que a ANOVA vai testar

A ANOVA de um fator responde:

As médias das medições são iguais em todos os locais?

Hipóteses

H0: todas as médias dos locais são iguais

H1: pelo menos uma média é diferente

**fator/grupo = Loca**

**variável resposta = Medição (µT)**

Bibliotecas

In [ ]:
import pandas as pd
from scipy.stats import f_oneway, levene
from statsmodels.stats.oneway import anova_oneway
import statsmodels.stats.multicomp as mc

Ler o arquivo

In [ ]:
df = pd.read_csv(
    "Trabalho_Ergonomia(DADOS).csv",
    sep=";",
    encoding="latin1",
    decimal=","
)

print(df.head())
print(df.columns)

   ID  Local  Medição  (µT)
0   1  PT1M4       1.061564
1   1  PT1M4       1.346322
2   1  PT1M4       1.054704
3   1  PT1M4       1.095514
4   1  PT1M4       1.159552
Index(['ID', 'Local', 'Medição  (µT)'], dtype='object')


In [ ]:
print(df["Local"].unique())
print(df["Local"].value_counts())

['PT1M4' 'PT2B7' 'PT3MM15' 'PT4A2' 'PT5A3' 'PT6CB3' 'PT7JO3' 'PT8BM16'
 'PT9CTBC' 'PT10BE12' 'PT11J1']
Local
PT5A3       26601
PT10BE12    22697
PT6CB3      22073
PT3MM15     14481
PT4A2       12821
PT9CTBC      9832
PT7JO3       9812
PT11J1       7429
PT1M4        4904
PT2B7        3849
PT8BM16      1269
Name: count, dtype: int64


Fazer uma análise descritiva inicial

count → quantidade de valores existentes na coluna
Ex.: se uma coluna tem 20 números preenchidos, então count = 20

mean → média aritmética dos valores
Fórmula: soma dos valores ÷ quantidade

std → desvio padrão
Mostra o quanto os valores estão espalhados em relação à média.
Quanto maior o std, mais dispersos os dados estão.

min → menor valor da coluna

max → maior valor da coluna

In [ ]:
resumo = df.groupby("Local")["Medição  (µT)"].agg(["count", "mean", "std", "min", "max"])
print(resumo)

          count      mean       std       min        max
Local                                                   
PT10BE12  22697  1.526769  0.329828  0.922376   2.422138
PT11J1     7429  1.185843  0.054020  1.007915   1.447908
PT1M4      4904  1.174102  0.127222  0.257572   1.813019
PT2B7      3849  1.179635  0.056902  1.023280   1.600298
PT3MM15   14481  1.129924  0.042390  0.975742   1.393654
PT4A2     12821  1.138249  0.058154  0.957409   1.469686
PT5A3     26601  1.219512  0.117418  1.074257   1.526593
PT6CB3    22073  1.349820  0.215203  0.975804   2.031427
PT7JO3     9812  1.166636  0.037931  1.044529   1.284339
PT8BM16    1269  1.120521  0.049885  1.001231   1.275308
PT9CTBC    9832  6.581233  3.262856  1.254425  15.282220


Separar os grupos

In [ ]:
grupos = [grupo["Medição  (µT)"].values for _, grupo in df.groupby("Local")]

Rodar a ANOVA

Como p < 0,05, rejeitamos H0.

Conclusão da ANOVA

Existe diferença estatisticamente significativa entre as médias das medições dos locais.

In [ ]:
f_stat, p_valor = f_oneway(*grupos)

print("Estatística F:", f_stat)
print("p-valor:", p_valor)

Estatística F: 32444.50261389591
p-valor: 0.0


Teste de Levene

In [ ]:
lev_stat, lev_p = levene(*grupos)

print("Estatística de Levene:", lev_stat)
print("p-valor de Levene:", lev_p)

Estatística de Levene: 23799.81221016461
p-valor de Levene: 0.0


Resultado:

Levene = 23799.81

p-valor = 0.0

Interpretação

Como p < 0,05, as variâncias dos grupos não são homogêneas.

Isso quer dizer que o pressuposto de variâncias iguais foi violado.

Nesse caso, o ideal é complementar com a ANOVA de Welch, que é mais robusta quando as variâncias são desiguais.

ANOVA de Welch

In [ ]:
resultado_welch = anova_oneway(
    df["Medição  (µT)"],
    groups=df["Local"],
    use_var="unequal"
)

print(resultado_welch)

statistic = 9133.822510351521
pvalue = 0.0
df = (10.0, np.float64(23973.356396064362))
df_num = 10.0
df_denom = 23973.356396064362
nobs_t = 135768.0
n_groups = 11
means = [1.52676852 1.18584262 1.17410163 1.17963545 1.12992354 1.1382488
     1.21951199 1.34982019 1.16663565 1.12052132 6.58123334]
nobs = [22697.  7429.  4904.  3849. 14481. 12821. 26601. 22073.  9812.  1269.
      9832.]
vars_ = [1.08786363e-01 2.91812006e-03 1.61854587e-02 3.23780727e-03
     1.79688861e-03 3.38188397e-03 1.37870040e-02 4.63121839e-02
     1.43877450e-03 2.48852892e-03 1.06462303e+01]
use_var = unequal
welch_correction = True
tuple = (np.float64(9133.822510351521), np.float64(0.0))


Resultado:

F de Welch = 9133.82

p-valor = 0.0

Conclusão de Welch

Mesmo considerando variâncias desiguais, o resultado continua altamente significativo.

Conclusão mais segura

As médias das medições diferem significativamente entre os locais.

A ANOVA só diz que há diferença em algum lugar.
Para saber entre quais locais, fazemos um pós-teste

Teste de Tukey

In [ ]:
comparacao = mc.pairwise_tukeyhsd(
    endog=df["Medição  (µT)"],
    groups=df["Local"],
    alpha=0.05
)

print(comparacao)

  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
 group1   group2 meandiff p-adj   lower   upper  reject
-------------------------------------------------------
PT10BE12  PT11J1  -0.3409    0.0 -0.3794 -0.3024   True
PT10BE12   PT1M4  -0.3527    0.0  -0.398 -0.3073   True
PT10BE12   PT2B7  -0.3471    0.0 -0.3973 -0.2969   True
PT10BE12 PT3MM15  -0.3968    0.0 -0.4275 -0.3662   True
PT10BE12   PT4A2  -0.3885    0.0 -0.4203 -0.3567   True
PT10BE12   PT5A3  -0.3073    0.0 -0.3333 -0.2812   True
PT10BE12  PT6CB3  -0.1769    0.0 -0.2042 -0.1497   True
PT10BE12  PT7JO3  -0.3601    0.0 -0.3949 -0.3253   True
PT10BE12 PT8BM16  -0.4062    0.0 -0.4893 -0.3232   True
PT10BE12 PT9CTBC   5.0545    0.0  5.0197  5.0892   True
  PT11J1   PT1M4  -0.0117 0.9998 -0.0647  0.0413  False
  PT11J1   PT2B7  -0.0062    1.0 -0.0634   0.051  False
  PT11J1 PT3MM15  -0.0559 0.0006  -0.097 -0.0148   True
  PT11J1   PT4A2  -0.0476 0.0119 -0.0896 -0.0056   True
  PT11J1   PT5A3   0.0337 0.1336 -0.0041  0.0715

Interpretação do pós-teste

O teste mostrou que:

PT9CTBC difere significativamente de todos os outros locais.

Vários pares entre os locais com médias próximas não apresentaram diferença significativa.

Exemplos de pares sem diferença significativa:

PT11J1 vs PT1M4

PT11J1 vs PT2B7

PT1M4 vs PT2B7

PT3MM15 vs PT4A2

PT4A2 vs PT7JO3

Ou seja:

existe uma diferença global forte;

parte dessa diferença é puxada principalmente por PT9CTBC;

muitos dos demais locais têm médias relativamente parecidas entre si.